# SQL analysis of Steam review patterns
Rebuild the local database from the supplied paid extract. The final query adds a CTE and window function for a specific question: how do first-listed genre groups rank within each price band? These groups contain at least 100 records. Rankings are descriptive.


In [1]:
import sqlite3
from pathlib import Path
import pandas as pd
games = pd.read_csv("games_clean.csv")
games["main_genre"] = games["Genres"].str.split(",").str[0]
conn = sqlite3.connect("steam.db")
games.to_sql("games",conn,if_exists="replace",index=False)
assert conn.execute("SELECT COUNT(*) FROM games").fetchone()[0] == len(games)
print("Loaded",len(games),"records")


Loaded 26471 records


In [2]:
query = '-- Run sql_analysis.ipynb first to load games_clean.csv into SQLite.\n-- main_genre is the first-listed source genre, not a verified primary genre.\n-- Scores are unweighted means across selected paid records.\n\n-- Review score across price bands\nSELECT price_band,\n       ROUND(AVG(review_score) * 100, 1) AS avg_score,\n       COUNT(*) AS num_games\nFROM games\nGROUP BY price_band\nORDER BY avg_score DESC'
print(pd.read_sql_query(query, conn).to_string(index=False))


price_band  avg_score  num_games
     $5-10       81.5       5901
    $10-20       80.8       3638
    $20-30       79.3        580
    $30-40       78.2        136
  Under $5       77.5      16102
    $40-70       75.2        114


In [3]:
query = '-- Review score by genre, restricting the comparison to groups with at least 100 records\nSELECT main_genre,\n       ROUND(AVG(review_score) * 100, 1) AS avg_score,\n       COUNT(*) AS num_games\nFROM games\nGROUP BY main_genre\nHAVING COUNT(*) >= 100\nORDER BY avg_score DESC'
print(pd.read_sql_query(query, conn).to_string(index=False))


main_genre  avg_score  num_games
 Adventure       81.4       6383
    Casual       80.3       3624
       RPG       78.9        632
     Indie       78.9       2791
  Strategy       77.9        522
    Action       77.8      11110
    Racing       74.7        209
Simulation       73.5        780


In [4]:
query = '-- Highly rated games under $10 with a substantial number of reviews\nSELECT Name,\n       Price,\n       ROUND(review_score * 100, 1) AS score,\n       total_reviews\nFROM games\nWHERE Price < 10\n  AND total_reviews >= 1000\nORDER BY review_score DESC\nLIMIT 10'
print(pd.read_sql_query(query, conn).to_string(index=False))


                               Name  Price  score  total_reviews
                        Kabuto Park   3.99   99.9           1036
               A Tower Full of Cats   4.19   99.5           1969
              A Castle Full of Cats   2.39   99.4           4009
         Dialtown: Phone Dating Sim   5.99   99.3           1954
                       The Upturned   5.99   99.3           2364
Aventura Copilului Albastru și Urât   1.33   99.3           2470
            Papa's Freezeria Deluxe   4.79   99.3          11314
                  Patrick's Parabox   9.99   99.2           4472
      The Void Rains Upon Her Heart   8.44   99.2           1658
            TOEM: A Photo Adventure   3.99   99.2           8385


In [5]:
query = '-- Rank first-listed genre groups within each price band (minimum 100 records).\n-- DENSE_RANK retains ties in the unrounded mean.\nWITH grouped AS (\n SELECT price_band, main_genre, COUNT(*) AS num_games, AVG(review_score) AS mean_score\n FROM games\n GROUP BY price_band, main_genre\n HAVING COUNT(*) >= 100\n)\nSELECT price_band, main_genre, num_games, ROUND(mean_score * 100, 2) AS avg_score,\n DENSE_RANK() OVER (PARTITION BY price_band ORDER BY mean_score DESC) AS rank_within_band\nFROM grouped\nORDER BY price_band, rank_within_band'
print(pd.read_sql_query(query, conn).to_string(index=False))


price_band main_genre  num_games  avg_score  rank_within_band
    $10-20  Adventure        866      84.76                 1
    $10-20        RPG        130      83.11                 2
    $10-20     Casual        289      81.66                 3
    $10-20      Indie        447      80.04                 4
    $10-20     Action       1557      79.54                 5
    $10-20 Simulation        180      76.35                 6
    $20-30  Adventure        102      84.78                 1
    $20-30     Action        250      76.57                 2
     $5-10  Adventure       1505      84.25                 1
     $5-10     Casual        622      82.80                 2
     $5-10   Strategy        133      82.12                 3
     $5-10     Action       2434      80.59                 4
     $5-10      Indie        709      80.39                 5
     $5-10        RPG        190      79.66                 6
     $5-10 Simulation        179      76.06                 7
  Under 

In [6]:
conn.close()